In [10]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg16, VGG16_Weights
import numpy as np
from tqdm.auto import tqdm

In [11]:
COMPETITION_ROOT = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'manifest.csv' in files:
        # manifest.csv is usually inside the 'dataset' subfolder
        COMPETITION_ROOT = os.path.dirname(root) 
        DATASET_ROOT = root
        break

if COMPETITION_ROOT is None:
    raise FileNotFoundError("Could not find competition data in /kaggle/input/")

# Add the competition root to Python's path so it can find the 'utility' folder
if COMPETITION_ROOT not in sys.path:
    sys.path.append(COMPETITION_ROOT)

print(f"Competition Root: {COMPETITION_ROOT}")
print(f"Dataset Root: {DATASET_ROOT}")

from utility.data import HackathonDataset, combined_roll_from_bundle
from utility.metric import score_item, leaderboard_score
from utility.submission import bundle_to_rows, write_submission


Competition Root: /kaggle/input/competitions/sapienza-genai-hackathon
Dataset Root: /kaggle/input/competitions/sapienza-genai-hackathon/dataset


In [12]:
class PianoRollDataset(Dataset):
    def __init__(self, dataset_obj, split="train"):
        self.dataset = dataset_obj
        self.split = split
        if split == "train":
            self.triplets = self.dataset.train()
        elif split == "val":
            self.triplets = self.dataset.val()
        else:
            self.triplets = self.dataset.test()
            
    def _bundle_to_tensor(self, bundle):
        pitched = combined_roll_from_bundle(bundle)
        return torch.tensor(np.stack([pitched, bundle["drum"]]), dtype=torch.float32)

    def __len__(self): return len(self.triplets)

    def __getitem__(self, idx):
        t = self.triplets[idx]
        load_Y = self.split in ["train", "val"]
        data_dict = self.dataset.get(t.item_id, load_Y=load_Y)
        
        X = self._bundle_to_tensor(data_dict["X"])
        Z = self._bundle_to_tensor(data_dict["Z"])
        
        if load_Y:
            Y = self._bundle_to_tensor(data_dict["Y"])
            return X, Z, Y, t.item_id
        return X, Z, t.item_id

In [21]:
class AdaIN(nn.Module):
    def forward(self, content, gamma, beta):
        b, c, h, w = content.size()
        c_std, c_mean = torch.std_mean(content, dim=[2, 3], keepdim=True)
        c_norm = (content - c_mean) / (c_std + 1e-5)
        return c_norm * gamma.view(b, c, 1, 1) + beta.view(b, c, 1, 1)

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, stride=2)]
        if normalize: layers.append(nn.InstanceNorm2d(out_c))
        layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class StyleTransferGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(2, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)
        self.enc4 = ConvBlock(128, 256)
        
        self.style_enc = nn.Sequential(
            ConvBlock(2, 32, False), ConvBlock(32, 64, False), 
            ConvBlock(64, 128, False), ConvBlock(128, 256, False),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc_gamma = nn.Linear(256, 256)
        self.fc_beta = nn.Linear(256, 256)
        self.adain = AdaIN()
        
        self.dec4 = nn.Sequential(nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.InstanceNorm2d(128), nn.ReLU(True))
        self.dec3 = nn.Sequential(nn.ConvTranspose2d(128 + 128, 64, 4, 2, 1), nn.InstanceNorm2d(64), nn.ReLU(True))
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(64 + 64, 32, 4, 2, 1), nn.InstanceNorm2d(32), nn.ReLU(True))
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(32 + 32, 2, 4, 2, 1), nn.Sigmoid())

    def _match_and_cat(self, upsampled, bypass):
        # Defensive check: if shapes don't match due to rounding, interpolate upsampled to bypass size
        if upsampled.shape[2:] != bypass.shape[2:]:
            upsampled = F.interpolate(upsampled, size=bypass.shape[2:], mode='bilinear', align_corners=False)
        return torch.cat([upsampled, bypass], dim=1)

    def forward(self, x, z):
        # Encoder
        x1 = self.enc1(x)
        x2 = self.enc2(x1)
        x3 = self.enc3(x2)
        x4 = self.enc4(x3)
        
        # Style
        s_feat = self.style_enc(z).view(-1, 256)
        adain_feat = self.adain(x4, self.fc_gamma(s_feat), self.fc_beta(s_feat))
        
        # Decoder with robust size-matching skip connections
        d4 = self.dec4(adain_feat)
        d3 = self.dec3(self._match_and_cat(d4, x3))
        d2 = self.dec2(self._match_and_cat(d3, x2))
        
        # Final output layer needs to match original input 'x' dimensions
        d1 = self.dec1(self._match_and_cat(d2, x1))
        if d1.shape[2:] != x.shape[2:]:
            d1 = F.interpolate(d1, size=x.shape[2:], mode='bilinear', align_corners=False)
            
        return d1

In [31]:
class MusicSparsityLoss(nn.Module):
    def __init__(self, note_weight=8.0):
        super().__init__()
        self.note_weight = note_weight
        
    def forward(self, y_hat, y_tgt):
        # Establish a binary mask where notes actively exist
        note_mask = (y_tgt > 0.05).float()
        
        # Heavy penalty for missing true notes
        note_loss = F.l1_loss(y_hat * note_mask, y_tgt * note_mask, reduction='sum') / (note_mask.sum() + 1e-8)
        # Standard penalty for quiet spaces
        silence_loss = F.l1_loss(y_hat * (1 - note_mask), y_tgt * (1 - note_mask), reduction='sum') / ((1 - note_mask).sum() + 1e-8)
        
        # Base MSE for overall structural cohesion
        mse_loss = F.mse_loss(y_hat, y_tgt)
        
        return mse_loss + (self.note_weight * note_loss) + silence_loss

In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FiLMResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, style_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        
        self.film_gamma = nn.Linear(style_dim, out_channels)
        self.film_beta = nn.Linear(style_dim, out_channels)

    def forward(self, x, style_emb):
        out = F.relu(self.bn1(self.conv1(x)))
        
        gamma = self.film_gamma(style_emb).unsqueeze(-1).unsqueeze(-1)
        beta = self.film_beta(style_emb).unsqueeze(-1).unsqueeze(-1)
        out = out * (1 + gamma) + beta
        
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class AttentionGate(nn.Module):
    def __init__(self, f_g, f_l, f_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(f_g, f_int, 1, bias=True), nn.BatchNorm2d(f_int))
        self.W_x = nn.Sequential(nn.Conv2d(f_l, f_int, 1, bias=True), nn.BatchNorm2d(f_int))
        self.psi = nn.Sequential(nn.Conv2d(f_int, 1, 1, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, g, x):
        if g.shape[2:] != x.shape[2:]:
            g = F.interpolate(g, size=x.shape[2:], mode='bilinear', align_corners=False)
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        return x * self.psi(psi)

class PianoRollResAttnNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Style Encoder
        self.style_enc = nn.Sequential(
            nn.Conv2d(2, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        # Content Encoder
        self.enc1 = FiLMResBlock(2, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = FiLMResBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = FiLMResBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = FiLMResBlock(128, 256)
        
        # Decoders
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = FiLMResBlock(256, 128)
        
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = FiLMResBlock(128, 64)
        
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = FiLMResBlock(64, 32)
        
        self.attn3 = AttentionGate(f_g=128, f_l=128, f_int=64)
        self.attn2 = AttentionGate(f_g=64, f_l=64, f_int=32)
        self.attn1 = AttentionGate(f_g=32, f_l=32, f_int=16)
        
        self.final_conv = nn.Sequential(nn.Conv2d(32, 2, 1), nn.Sigmoid())

    def forward(self, x, z):
        # Extract Style Latent Vector
        style_emb = self.style_enc(z).view(-1, 256)
        
        # Content Encoder
        x1 = self.enc1(x, style_emb)
        x2 = self.enc2(self.pool1(x1), style_emb)
        x3 = self.enc3(self.pool2(x2), style_emb)
        
        # Bottleneck
        b = self.bottleneck(self.pool3(x3), style_emb)
        
        # Decoder 3
        g3 = self.up3(b)
        x3_attn = self.attn3(g3, x3)
        if g3.shape[2:] != x3_attn.shape[2:]:
            g3 = F.interpolate(g3, size=x3_attn.shape[2:], mode='bilinear', align_corners=False)
        d3 = self.dec3(torch.cat([g3, x3_attn], dim=1), style_emb)
        
        # Decoder 2
        g2 = self.up2(d3)
        x2_attn = self.attn2(g2, x2)
        if g2.shape[2:] != x2_attn.shape[2:]:
            g2 = F.interpolate(g2, size=x2_attn.shape[2:], mode='bilinear', align_corners=False)
        d2 = self.dec2(torch.cat([g2, x2_attn], dim=1), style_emb)
        
        # Decoder 1
        g1 = self.up1(d2)
        x1_attn = self.attn1(g1, x1)
        if g1.shape[2:] != x1_attn.shape[2:]:
            g1 = F.interpolate(g1, size=x1_attn.shape[2:], mode='bilinear', align_corners=False)
        d1 = self.dec1(torch.cat([g1, x1_attn], dim=1), style_emb)
        
        out = self.final_conv(d1)
        if out.shape[2:] != x.shape[2:]:
            out = F.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)
        return out

In [24]:
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
        self.blocks = nn.ModuleList([vgg[:4], vgg[4:9], vgg[9:16]]).eval()
        for p in self.parameters(): p.requires_grad = False

    def forward(self, y_hat, y_tgt, z_style):
        # FORCE FP32: Disable autocast locally to prevent Gram Matrix overflow
        with torch.amp.autocast('cuda', enabled=False):
            y_hat = y_hat.float()
            y_tgt = y_tgt.float()
            z_style = z_style.float()
            
            def to_3c(t): return torch.cat([t, t[:, 1:2, :, :]], dim=1)
            x_hat, x_tgt, x_z = to_3c(y_hat), to_3c(y_tgt), to_3c(z_style)
            
            l_perc, l_style = 0.0, 0.0
            for block in self.blocks:
                x_hat, x_tgt, x_z = block(x_hat), block(x_tgt), block(x_z)
                l_perc += F.l1_loss(x_hat, x_tgt)
                
                b, c, h, w = x_hat.shape
                feat_hat = x_hat.view(b, c, h*w)
                feat_z = x_z.view(b, c, h*w)
                
                # Multiplication happens safely in float32 now
                gram_hat = torch.bmm(feat_hat, feat_hat.transpose(1, 2)) / (c*h*w)
                gram_z = torch.bmm(feat_z, feat_z.transpose(1, 2)) / (c*h*w)
                l_style += F.mse_loss(gram_hat, gram_z)
                
            return l_perc, l_style

In [15]:
def validate_model(model, dataset_obj, val_loader, device):
    model.eval()
    all_scores = []
    
    with torch.no_grad():
        for X_tensor, Z_tensor, _, item_ids in val_loader:
            X_tensor, Z_tensor = X_tensor.to(device), Z_tensor.to(device)
            Y_hat_tensor = model(X_tensor, Z_tensor)
            
            for i in range(len(item_ids)):
                item_id = item_ids[i]
                y_out = Y_hat_tensor[i].cpu().numpy()
                
                output_bundle = {
                    "pitched": np.expand_dims(y_out[0], axis=0),
                    "drum": y_out[1],
                    "track_ids": np.array([0], dtype=np.int32)
                }
                
                raw_data = dataset_obj.get(item_id, load_Y=False)
                content_roll = combined_roll_from_bundle(raw_data["X"])
                target_profile = raw_data["style_profile"]
                
                # Calculate scores
                item_score_dict = score_item(content_roll, output_bundle, target_profile)
                all_scores.append(item_score_dict)

    mean_score = leaderboard_score(all_scores)
    mean_cp = np.mean([d["CP"] for d in all_scores])
    mean_sf = np.mean([d["SF"] for d in all_scores])
    
    return mean_score, mean_cp, mean_sf

In [38]:
def train_model(dataset_root, epochs=20, batch_size=32, lr=4e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    hack_data = HackathonDataset(dataset_root)
    train_ds = PianoRollDataset(hack_data, split="train")
    val_ds = PianoRollDataset(hack_data, split="val")
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=4, pin_memory=True)
    
    model = PianoRollResAttnNet()
    if torch.cuda.device_count() > 1:
        print(f"🔥 Blast off! Dual GPU active. Splitting {batch_size} batch size across 2 T4 cards.")
        model = nn.DataParallel(model)
    model = model.to(device)
    
    criterion = MusicSparsityLoss(note_weight=8.0).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    
    warmup_epochs = 2
    scheduler_warmup = LinearLR(optimizer, start_factor=0.2, end_factor=1.0, total_iters=warmup_epochs)
    scheduler_cosine = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
    scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[warmup_epochs])
    
    scaler = torch.amp.GradScaler('cuda')
    best_val_score = 0.0
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        loop = tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}/{epochs}")
        
        for X, Z, Y, _ in loop:
            X = X.to(device, non_blocking=True)
            Z = Z.to(device, non_blocking=True)
            Y = Y.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            
            # FIXED: Unified global autocast block handles forward and loss types seamlessly
            with torch.amp.autocast('cuda'):
                Y_hat = model(X, Z)
                loss = criterion(Y_hat, Y)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
            loop.set_postfix(loss=loss.item(), lr=optimizer.param_groups[0]['lr'])
            
        scheduler.step()
        avg_train_loss = epoch_loss / len(train_loader)
        
        val_score, val_cp, val_sf = validate_model(model, hack_data, val_loader, device)
        print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | "
              f"Val Score: {val_score:.4f} (CP: {val_cp:.4f}, SF: {val_sf:.4f})")
        
        if val_score > best_val_score:
            best_val_score = val_score
            actual_model = model.module if isinstance(model, nn.DataParallel) else model
            torch.save(actual_model.state_dict(), "best_model.pth")
            print("   -> 🌟 New Best Score! Saved best_model.pth")
            
    return model, hack_data

In [17]:
def generate_submission(model, hack_data, output_csv="submission.csv"):
    device = next(model.parameters()).device
    model.eval()
    
    test_ds = PianoRollDataset(hack_data, split="test")
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
    
    all_rows = []
    print("\nGenerating Kaggle submission...")
    with torch.no_grad():
        for X, Z, item_id in tqdm(test_loader):
            X, Z = X.to(device), Z.to(device)
            Y_hat = model(X, Z)
            
            y_out = Y_hat.squeeze(0).cpu().numpy()
            bundle = {
                "pitched": np.expand_dims(y_out[0], axis=0),
                "drum": y_out[1],
                "track_ids": np.array([0], dtype=np.int32)
            }
            
            item_rows = list(bundle_to_rows(item_id[0], bundle))
            if len(item_rows) == 0:
                 all_rows.append({"item_id": item_id[0], "notes": ""}) 
            else:
                 all_rows.extend(item_rows)
            
    # Output saves directly to Kaggle's working directory /kaggle/working/
    write_submission(output_csv, all_rows)
    print(f"✅ Saved to {output_csv}. Submit this output file!")

In [40]:
trained_model, dataset = train_model(DATASET_ROOT, epochs=20, batch_size=32, lr=2e-4)

🔥 Blast off! Dual GPU active. Splitting 16 batch size across 2 T4 cards.


Epoch 1/20:   0%|          | 0/359 [00:00<?, ?it/s]

Epoch 01 | Train Loss: 1.1103 | Val Score: 0.1254 (CP: 0.4255, SF: 0.0769)
   -> 🌟 New Best Score! Saved best_model.pth


Epoch 2/20:   0%|          | 0/359 [00:00<?, ?it/s]

Epoch 02 | Train Loss: 0.5571 | Val Score: 0.0846 (CP: 0.4637, SF: 0.0483)


...

Epoch 18 | Train Loss: 0.0475 | Val Score: 0.5857 (CP: 0.9197, SF: 0.4374)
   -> 🌟 New Best Score! Saved best_model.pth


Epoch 19/20:   0%|          | 0/359 [00:00<?, ?it/s]

Epoch 19 | Train Loss: 0.0462 | Val Score: 0.5860 (CP: 0.9204, SF: 0.4375)
   -> 🌟 New Best Score! Saved best_model.pth


Epoch 20/20:   0%|          | 0/359 [00:01<?, ?it/s]

Epoch 20 | Train Loss: 0.0463 | Val Score: 0.5861 (CP: 0.9200, SF: 0.4377)
   -> 🌟 New Best Score! Saved best_model.pth


In [41]:
generate_submission(trained_model, dataset, output_csv="submission.csv")


Generating Kaggle submission...


  0%|          | 0/1200 [00:00<?, ?it/s]

✅ Saved to submission.csv. Submit this output file!
